In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/itzmir/onehealth-processed/dyslipidemia_X_test.npy
/kaggle/input/datasets/itzmir/onehealth-processed/cvd_X_test.npy
/kaggle/input/datasets/itzmir/onehealth-processed/external_full_curated.csv
/kaggle/input/datasets/itzmir/onehealth-processed/hypertension_y_train.npy
/kaggle/input/datasets/itzmir/onehealth-processed/diabetes_X_test.npy
/kaggle/input/datasets/itzmir/onehealth-processed/dyslipidemia_scaler.pkl
/kaggle/input/datasets/itzmir/onehealth-processed/diabetes_y_train.npy
/kaggle/input/datasets/itzmir/onehealth-processed/cvd_y_test.npy
/kaggle/input/datasets/itzmir/onehealth-processed/hypertension_X_train.npy
/kaggle/input/datasets/itzmir/onehealth-processed/dyslipidemia_y_train.npy
/kaggle/input/datasets/itzmir/onehealth-processed/cvd_y_train.npy
/kaggle/input/datasets/itzmir/onehealth-processed/ckd_X_train.npy
/kaggle/input/datasets/itzmir/onehealth-processed/metabolic_syndrome_X_train.npy
/kaggle/input/datasets/itzmir/onehealth-processed/liver_disease_y_t

In [6]:
import os, shutil

SRC_PROCESSED = "/kaggle/input/datasets/itzmir/onehealth-processed"
SRC_MODEL = "/kaggle/input/datasets/itzmir/onehealth-model"

DST_PROCESSED = "/kaggle/working/processed"
os.makedirs(DST_PROCESSED, exist_ok=True)

# Copy all files (Kaggle input is read-only; working is writable)
for f in os.listdir(SRC_PROCESSED):
    shutil.copy(os.path.join(SRC_PROCESSED, f), os.path.join(DST_PROCESSED, f))

print("Files copied:", len(os.listdir(DST_PROCESSED)))
print("Model present:", os.path.exists(f"{SRC_MODEL}/onehealth_net_final.pt"))

Files copied: 43
Model present: True


In [7]:
!pip install -q pytorch-tabnet==4.1.0 rtdl_revisiting_models delu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.6 MB/s eta 0:00:00


In [9]:
import numpy as np, pandas as pd, json, pickle, os
import torch, torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

DRIVE_DIR = "/kaggle/working"
PROCESSED = "/kaggle/working/processed"
MODEL_PATH = "/kaggle/input/datasets/itzmir/onehealth-model/onehealth_net_final.pt"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DISEASES = ["diabetes","hypertension","ckd","dyslipidemia",
            "cvd","metabolic_syndrome","liver_disease"]
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
if DEVICE.type == "cuda": torch.cuda.manual_seed_all(SEED)
print("Device:", DEVICE)

with open(f"{PROCESSED}/feature_names.json") as f: FEATURE_NAMES = json.load(f)
with open(f"{PROCESSED}/leakage_map.json") as f: LEAKAGE_MAP = json.load(f)

master = pd.read_csv(f"{PROCESSED}/labeled_master_all_cycles.csv")
all_features = set()
for label, feats in FEATURE_NAMES.items(): all_features.update(feats)
all_features = sorted([f for f in all_features if f in master.columns])

label_cols = [f"label_{d}" for d in DISEASES]
mask = master[label_cols].notna().any(axis=1)
X_raw = master.loc[mask, all_features].apply(pd.to_numeric, errors="coerce").values.astype(np.float32)

LEAK_MASK = np.zeros((len(DISEASES), len(all_features)), dtype=bool)
for i, d in enumerate(DISEASES):
    leaked = set(LEAKAGE_MAP.get(d, []))
    for j, f in enumerate(all_features):
        if f in leaked: LEAK_MASK[i, j] = True

idx_tr, idx_val = train_test_split(np.arange(X_raw.shape[0]), test_size=0.2,
                                    random_state=SEED, shuffle=True)
imp = SimpleImputer(strategy="median").fit(X_raw[idx_tr])
sc  = StandardScaler().fit(imp.transform(X_raw[idx_tr]))

class OneHealthNet(nn.Module):
    def __init__(self, n_features, n_diseases, leak_mask, d_model=96, n_heads=6,
                 n_layers=2, dropout=0.15):
        super().__init__()
        self.n_features, self.n_diseases = n_features, n_diseases
        self.register_buffer("leak_mask", torch.tensor(leak_mask, dtype=torch.bool))
        self.tok_weight = nn.Parameter(torch.randn(n_features, d_model) * 0.02)
        self.tok_bias   = nn.Parameter(torch.zeros(n_features, d_model))
        self.mask_embed = nn.Parameter(torch.randn(d_model) * 0.02)
        el = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
                                        dim_feedforward=d_model*4, dropout=dropout,
                                        batch_first=True, activation="gelu", norm_first=True)
        self.encoder = nn.TransformerEncoder(el, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.disease_queries = nn.Parameter(torch.randn(n_diseases, d_model) * 0.02)
        self.pool_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.heads = nn.ModuleList([nn.Linear(d_model, 1) for _ in range(n_diseases)])
        self.aux_head = nn.Linear(d_model, 1)
    def forward(self, x):
        B, F = x.shape; D = self.n_diseases
        base = x.unsqueeze(-1) * self.tok_weight.unsqueeze(0) + self.tok_bias.unsqueeze(0)
        tokens = base.unsqueeze(1).expand(B, D, F, -1).clone()
        m = self.leak_mask.unsqueeze(0).expand(B, D, F)
        tokens[m] = self.mask_embed
        tokens = tokens.reshape(B*D, F, -1)
        h = self.norm(self.encoder(tokens))
        q = self.disease_queries.unsqueeze(0).expand(B, D, -1).reshape(B*D, 1, -1)
        pooled, _ = self.pool_attn(q, h, h)
        pooled = pooled.reshape(B, D, -1)
        logits = torch.stack([self.heads[i](pooled[:, i]).squeeze(-1) for i in range(D)], dim=1)
        return logits, self.aux_head(pooled.mean(dim=1))

model = OneHealthNet(len(all_features), len(DISEASES), LEAK_MASK).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()
print("✅ Loaded OneHealth-Net")

ext = pd.read_csv(f"{PROCESSED}/external_full_curated.csv")
X_ext = np.full((ext.shape[0], len(all_features)), np.nan, dtype=np.float32)
for j, f in enumerate(all_features):
    if f in ext.columns:
        X_ext[:, j] = pd.to_numeric(ext[f], errors="coerce").values
X_ext = sc.transform(imp.transform(X_ext)).astype(np.float32)
Y_ext = ext[label_cols].values.astype(np.float32)
Y_ext_present = (~np.isnan(Y_ext)).astype(np.float32)
Y_ext = np.nan_to_num(Y_ext, nan=0.0)

with torch.no_grad():
    logits, _ = model(torch.tensor(X_ext, device=DEVICE))
    prob_all = torch.sigmoid(logits).cpu().numpy()

PERM_PATH = f"{DRIVE_DIR}/permutation_importance.pkl"
if os.path.exists(PERM_PATH):
    with open(PERM_PATH, "rb") as f:
        perm_results = pickle.load(f)
    print("Existing:", list(perm_results.keys()))
else:
    perm_results = {}

N_PERM = 5
for i, d in enumerate(DISEASES):
    if d in perm_results:
        print(f"{d}: already done")
        continue

    print(f"\n=== Processing {d} ===")
    lab = Y_ext_present[:, i] == 1
    X_use = X_ext[lab]; y_use = Y_ext[lab, i]; base_prob = prob_all[lab, i]
    base_auroc = roc_auc_score(y_use, base_prob)

    rng = np.random.default_rng(SEED + i)
    imp_d = {}
    for j in range(X_use.shape[1]):
        drops = []
        for _ in range(N_PERM):
            Xp = X_use.copy()
            Xp[:, j] = rng.permutation(Xp[:, j])
            with torch.no_grad():
                lo, _ = model(torch.tensor(Xp, device=DEVICE))
                pp = torch.sigmoid(lo).cpu().numpy()[:, i]
            drops.append(base_auroc - roc_auc_score(y_use, pp))
        imp_d[all_features[j]] = float(np.mean(drops))

    perm_results[d] = imp_d
    with open(PERM_PATH, "wb") as f:
        pickle.dump(perm_results, f)
    print(f"  ✅ Saved (baseline AUROC={base_auroc:.4f})")
    top10 = sorted(imp_d.items(), key=lambda x: -x[1])[:10]
    for fname, drop in top10:
        print(f"    {fname:25s} {drop:+.4f}")

print("\n🎉 Done. Diseases:", list(perm_results.keys()))

Device: cuda


/tmp/ipykernel_58/502397905.py:55: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(el, num_layers=n_layers)


✅ Loaded OneHealth-Net

=== Processing diabetes ===
  ✅ Saved (baseline AUROC=0.9335)
    LBXSOSSI                  +0.2821
    LBXSNASI                  +0.2272
    LBXSBU                    +0.0845
    RIDAGEYR                  +0.0306
    LBXTC                     +0.0117
    BMXWAIST                  +0.0048
    LBXHGB                    +0.0047
    LBXIN                     +0.0027
    RIDRETH3                  +0.0023
    URDACT                    +0.0022

=== Processing hypertension ===
  ✅ Saved (baseline AUROC=0.7971)
    RIDAGEYR                  +0.1095
    LBXSNASI                  +0.0068
    URDACT                    +0.0063
    LBXSGTSI                  +0.0062
    LBXSUA                    +0.0057
    LBXSOSSI                  +0.0056
    BMXBMI                    +0.0046
    LBXIN                     +0.0043
    BMXWT                     +0.0042
    RIDRETH3                  +0.0042

=== Processing ckd ===
  ✅ Saved (baseline AUROC=0.8759)
    LBXSBU                   

In [4]:
import json

with open("/kaggle/working/processed/leakage_map.json") as f:
    lm = json.load(f)

use_bare = "diabetes" in lm and "label_diabetes" not in lm
key = (lambda d: d) if use_bare else (lambda d: f"label_{d}")
print(f"Key format: {'bare' if use_bare else 'label_ prefixed'}")

NEW_LEAKS = ["LBXSOSSI", "LBXSNASI", "LBXSBU"]

for disease in ["diabetes", "metabolic_syndrome"]:
    k = key(disease)
    existing = set(lm.get(k, []))
    for feat in NEW_LEAKS:
        if feat not in existing:
            lm[k].append(feat)
    print(f"{disease}: {len(lm[k])} total leaks (was {len(existing)})")

with open("/kaggle/working/processed/leakage_map.json", "w") as f:
    json.dump(lm, f, indent=2)
print("✅ leakage_map.json updated")

Key format: label_ prefixed
diabetes: 13 total leaks (was 10)
metabolic_syndrome: 22 total leaks (was 19)
✅ leakage_map.json updated


In [5]:
with open("/kaggle/working/processed/leakage_map.json") as f:
    lm = json.load(f)

use_bare = "diabetes" in lm
def get_leaks(d):
    return lm.get(d if use_bare else f"label_{d}", [])

for d in ["diabetes", "metabolic_syndrome", "hypertension"]:
    print(f"{d}:")
    for f in get_leaks(d):
        print(f"  {f}")
    print()

diabetes:
  LBXGH
  LBXGLU
  DIQ010
  HOMA_IR
  LBXSGL
  met_syndrome_count
  ASCVD_risk
  DIQ160
  DID040
  DIQ080
  LBXSOSSI
  LBXSNASI
  LBXSBU

metabolic_syndrome:
  BMXWAIST
  LBXTR
  LBDHDD
  BPXSY1
  BPXSY2
  BPXSY3
  BPXSY4
  BPXDI1
  BPXDI2
  BPXDI3
  BPXDI4
  mean_sbp
  mean_dbp
  LBXGLU
  WHR
  LBXSGL
  LBXSTR
  met_syndrome_count
  ASCVD_risk
  LBXSOSSI
  LBXSNASI
  LBXSBU

hypertension:
  BPXSY1
  BPXSY2
  BPXSY3
  BPXSY4
  BPXDI1
  BPXDI2
  BPXDI3
  BPXDI4
  BPXPLS
  BPXPULS
  BPQ020
  mean_sbp
  mean_dbp
  ASCVD_risk



In [6]:
# === Retrain OneHealth-Net with corrected leakage map ===
import numpy as np, pandas as pd, json, pickle, os, copy
import torch, torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, matthews_corrcoef, roc_curve

PROCESSED = "/kaggle/working/processed"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DISEASES = ["diabetes","hypertension","ckd","dyslipidemia","cvd","metabolic_syndrome","liver_disease"]
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
if DEVICE.type == "cuda": torch.cuda.manual_seed_all(SEED)
print("Device:", DEVICE)

with open(f"{PROCESSED}/feature_names.json") as f: FEATURE_NAMES = json.load(f)
with open(f"{PROCESSED}/leakage_map.json") as f: LEAKAGE_MAP = json.load(f)

master = pd.read_csv(f"{PROCESSED}/labeled_master_all_cycles.csv")
all_features = set()
for label, feats in FEATURE_NAMES.items(): all_features.update(feats)
all_features = sorted([f for f in all_features if f in master.columns])
print(f"Union features: {len(all_features)}")

label_cols = [f"label_{d}" for d in DISEASES]
mask = master[label_cols].notna().any(axis=1)
X_raw = master.loc[mask, all_features].apply(pd.to_numeric, errors="coerce").values.astype(np.float32)
Y_raw = master.loc[mask, label_cols].values.astype(np.float32)
Y_present = (~np.isnan(Y_raw)).astype(np.float32)
Y_mt = np.nan_to_num(Y_raw, nan=0.0)

# Auxiliary target
m = master.loc[mask]
comp = (
    (((m["RIAGENDR"] == 1) & (m["BMXWAIST"] > 102)) | ((m["RIAGENDR"] == 2) & (m["BMXWAIST"] > 88))).astype(int)
    + (m["LBXTR"].fillna(0) >= 150).astype(int)
    + (((m["RIAGENDR"] == 1) & (m["LBDHDD"].fillna(100) < 40)) | ((m["RIAGENDR"] == 2) & (m["LBDHDD"].fillna(100) < 50))).astype(int)
    + ((m["mean_sbp"].fillna(0) >= 130) | (m["mean_dbp"].fillna(0) >= 85)).astype(int)
    + (m["LBXGLU"].fillna(0) >= 100).astype(int)
)
aux_all = comp.values.astype(np.float32)

# Leakage mask
use_bare = "diabetes" in LEAKAGE_MAP
def get_leaks(d):
    return LEAKAGE_MAP.get(d if use_bare else f"label_{d}", [])

LEAK_MASK = np.zeros((len(DISEASES), len(all_features)), dtype=bool)
for i, d in enumerate(DISEASES):
    leaked = set(get_leaks(d))
    for j, f in enumerate(all_features):
        if f in leaked: LEAK_MASK[i, j] = True

print("\nHidden feature counts (should show increased for diabetes + metabolic_syndrome):")
for i, d in enumerate(DISEASES):
    print(f"  {d:22s} {int(LEAK_MASK[i].sum())}")

# Model class
class OneHealthNet(nn.Module):
    def __init__(self, n_features, n_diseases, leak_mask, d_model=96, n_heads=6, n_layers=2, dropout=0.15):
        super().__init__()
        self.n_features, self.n_diseases = n_features, n_diseases
        self.register_buffer("leak_mask", torch.tensor(leak_mask, dtype=torch.bool))
        self.tok_weight = nn.Parameter(torch.randn(n_features, d_model) * 0.02)
        self.tok_bias   = nn.Parameter(torch.zeros(n_features, d_model))
        self.mask_embed = nn.Parameter(torch.randn(d_model) * 0.02)
        el = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=d_model*4,
                                        dropout=dropout, batch_first=True, activation="gelu", norm_first=True)
        self.encoder = nn.TransformerEncoder(el, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.disease_queries = nn.Parameter(torch.randn(n_diseases, d_model) * 0.02)
        self.pool_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.heads = nn.ModuleList([nn.Linear(d_model, 1) for _ in range(n_diseases)])
        self.aux_head = nn.Linear(d_model, 1)

    def forward(self, x):
        B, F = x.shape; D = self.n_diseases
        base = x.unsqueeze(-1) * self.tok_weight.unsqueeze(0) + self.tok_bias.unsqueeze(0)
        tokens = base.unsqueeze(1).expand(B, D, F, -1).clone()
        m = self.leak_mask.unsqueeze(0).expand(B, D, F)
        tokens[m] = self.mask_embed
        tokens = tokens.reshape(B*D, F, -1)
        h = self.norm(self.encoder(tokens))
        q = self.disease_queries.unsqueeze(0).expand(B, D, -1).reshape(B*D, 1, -1)
        pooled, _ = self.pool_attn(q, h, h)
        pooled = pooled.reshape(B, D, -1)
        logits = torch.stack([self.heads[i](pooled[:, i]).squeeze(-1) for i in range(D)], dim=1)
        return logits, self.aux_head(pooled.mean(dim=1))

# Split + scale
idx_tr, idx_val = train_test_split(np.arange(X_raw.shape[0]), test_size=0.2, random_state=SEED, shuffle=True)
imp = SimpleImputer(strategy="median").fit(X_raw[idx_tr])
sc  = StandardScaler().fit(imp.transform(X_raw[idx_tr]))
X_tr  = sc.transform(imp.transform(X_raw[idx_tr])).astype(np.float32)
X_val = sc.transform(imp.transform(X_raw[idx_val])).astype(np.float32)
Y_tr, Y_val = Y_mt[idx_tr], Y_mt[idx_val]
P_tr, P_val = Y_present[idx_tr], Y_present[idx_val]
aux_tr = np.nan_to_num(aux_all[idx_tr], nan=0.0)
aux_present_tr = (~np.isnan(aux_all[idx_tr])).astype(np.float32)

pos_weight = []
for i in range(len(DISEASES)):
    lab = P_tr[:, i] == 1
    pos = Y_tr[lab, i].sum(); neg = lab.sum() - pos
    pos_weight.append(float(neg / max(pos, 1)))
pos_weight = torch.tensor(pos_weight, device=DEVICE)
print(f"\npos_weight: {[round(w, 2) for w in pos_weight.tolist()]}")

# Setup training
model = OneHealthNet(len(all_features), len(DISEASES), LEAK_MASK).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
ema_model = copy.deepcopy(model)
for p in ema_model.parameters(): p.requires_grad_(False)
EMA_DECAY = 0.995

def update_ema():
    with torch.no_grad():
        for ep, p in zip(ema_model.parameters(), model.parameters()):
            ep.mul_(EMA_DECAY).add_(p, alpha=1 - EMA_DECAY)
        for eb, b in zip(ema_model.buffers(), model.buffers()): eb.copy_(b)

train_ds = TensorDataset(torch.tensor(X_tr), torch.tensor(Y_tr), torch.tensor(P_tr),
                          torch.tensor(aux_tr), torch.tensor(aux_present_tr))
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, drop_last=True)

N_EPOCHS, WARMUP, PATIENCE = 200, 10, 40
LABEL_SMOOTH, AUX_WEIGHT = 0.02, 0.1
steps_per_epoch = len(train_loader)

def lr_lambda(step):
    ep = step / steps_per_epoch
    if ep < WARMUP: return ep / WARMUP
    prog = (ep - WARMUP) / max(N_EPOCHS - WARMUP, 1)
    return 0.5 * (1 + np.cos(np.pi * min(prog, 1.0)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
def smooth(y, eps=LABEL_SMOOTH): return y * (1 - eps) + 0.5 * eps

best, counter, best_state, best_is_ema = 0, 0, None, False

@torch.no_grad()
def val_aurocs(m):
    m.eval()
    logits, _ = m(torch.tensor(X_val, device=DEVICE))
    prob = torch.sigmoid(logits).cpu().numpy()
    return [roc_auc_score(Y_val[P_val[:, i] == 1, i], prob[P_val[:, i] == 1, i]) for i in range(len(DISEASES))], prob

print("\nStarting training...")
for epoch in range(N_EPOCHS):
    model.train(); total_loss, total_n = 0, 0
    for xb, yb, pb, axb, apb in train_loader:
        xb, yb, pb = xb.to(DEVICE), yb.to(DEVICE), pb.to(DEVICE)
        axb, apb = axb.to(DEVICE), apb.to(DEVICE)
        optimizer.zero_grad()
        logits, aux_pred = model(xb)
        loss_mat = F.binary_cross_entropy_with_logits(logits, smooth(yb), pos_weight=pos_weight, reduction="none")
        main_loss = (loss_mat * pb).sum() / pb.sum().clamp(min=1)
        aux_loss = (((aux_pred.squeeze(-1) - axb) ** 2) * apb).sum() / apb.sum().clamp(min=1)
        loss = main_loss + AUX_WEIGHT * aux_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step(); update_ema()
        total_loss += loss.item() * xb.size(0); total_n += xb.size(0)

    raw_aurocs, _ = val_aurocs(model)
    ema_aurocs, _ = val_aurocs(ema_model)
    raw_mean, ema_mean = float(np.mean(raw_aurocs)), float(np.mean(ema_aurocs))
    cur_mean, cur_is_ema = (ema_mean, True) if ema_mean >= raw_mean else (raw_mean, False)
    if cur_mean > best:
        best, counter, best_is_ema = cur_mean, 0, cur_is_ema
        src = ema_model if cur_is_ema else model
        best_state = {k: v.cpu().clone() for k, v in src.state_dict().items()}
    else:
        counter += 1
        if counter >= PATIENCE:
            print(f"Early stop at epoch {epoch}"); break
    if epoch % 5 == 0:
        print(f"Epoch {epoch:3d} loss={total_loss/total_n:.4f} raw={raw_mean:.4f} ema={ema_mean:.4f} lr={scheduler.get_last_lr()[0]:.2e}")

final_model = ema_model if best_is_ema else model
final_model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
print(f"\nBest mean AUROC={best:.4f} (used {'EMA' if best_is_ema else 'raw'})")

# Per-disease final
_, val_prob = val_aurocs(final_model)
print("\n=== Corrected per-disease validation ===")
for i, d in enumerate(DISEASES):
    lab = P_val[:, i] == 1
    prob = val_prob[lab, i]; y = Y_val[lab, i]
    fpr, tpr, thr = roc_curve(y, prob)
    best_t = float(thr[int(np.argmax(tpr-fpr))])
    y_pred = (prob >= best_t).astype(int)
    print(f"  {d:22s} AUROC={roc_auc_score(y, prob):.4f}  AUPRC={average_precision_score(y, prob):.4f}  "
          f"F1={f1_score(y, y_pred):.4f}  MCC={matthews_corrcoef(y, y_pred):.4f}")

# Save
torch.save(final_model.state_dict(), "/kaggle/working/onehealth_net_v3_leakfix.pt")
print(f"\n✅ Model saved to /kaggle/working/onehealth_net_v3_leakfix.pt")

Device: cuda
Union features: 104

Hidden feature counts (should show increased for diabetes + metabolic_syndrome):
  diabetes               8
  hypertension           10
  ckd                    4
  dyslipidemia           3
  cvd                    7
  metabolic_syndrome     15
  liver_disease          4

pos_weight: [4.26, 0.9, 10.09, 0.61, 7.75, 1.91, 5.77]


/tmp/ipykernel_58/3648302846.py:72: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(el, num_layers=n_layers)



Starting training...
Epoch   0 loss=1.2587 raw=0.7494 ema=0.5459 lr=1.00e-04
Epoch   5 loss=0.7793 raw=0.8209 ema=0.8114 lr=6.00e-04
Epoch  10 loss=0.7490 raw=0.8270 ema=0.8306 lr=1.00e-03
Epoch  15 loss=0.7207 raw=0.8312 ema=0.8349 lr=9.98e-04
Epoch  20 loss=0.7077 raw=0.8301 ema=0.8352 lr=9.92e-04
Epoch  25 loss=0.6978 raw=0.8300 ema=0.8342 lr=9.83e-04
Epoch  30 loss=0.6827 raw=0.8312 ema=0.8325 lr=9.70e-04
Epoch  35 loss=0.6693 raw=0.8222 ema=0.8306 lr=9.55e-04
Epoch  40 loss=0.6582 raw=0.8253 ema=0.8274 lr=9.36e-04
Epoch  45 loss=0.6410 raw=0.8224 ema=0.8236 lr=9.14e-04
Epoch  50 loss=0.6293 raw=0.8181 ema=0.8199 lr=8.89e-04
Epoch  55 loss=0.6142 raw=0.8127 ema=0.8193 lr=8.62e-04
Early stop at epoch 57

Best mean AUROC=0.8355 (used EMA)

=== Corrected per-disease validation ===
  diabetes               AUROC=0.8681  AUPRC=0.6470  F1=0.5899  MCC=0.4874
  hypertension           AUROC=0.8275  AUPRC=0.8276  F1=0.7584  MCC=0.5149
  ckd                    AUROC=0.9003  AUPRC=0.6475  F1=

In [7]:
# === Verify the leak is gone ===
import numpy as np, pandas as pd, json
import torch, torch.nn as nn
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

PROCESSED = "/kaggle/working/processed"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DISEASES = ["diabetes","hypertension","ckd","dyslipidemia","cvd","metabolic_syndrome","liver_disease"]
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)

with open(f"{PROCESSED}/feature_names.json") as f: FEATURE_NAMES = json.load(f)
with open(f"{PROCESSED}/leakage_map.json") as f: LEAKAGE_MAP = json.load(f)

master = pd.read_csv(f"{PROCESSED}/labeled_master_all_cycles.csv")
all_features = set()
for label, feats in FEATURE_NAMES.items(): all_features.update(feats)
all_features = sorted([f for f in all_features if f in master.columns])

label_cols = [f"label_{d}" for d in DISEASES]
mask = master[label_cols].notna().any(axis=1)
X_raw = master.loc[mask, all_features].apply(pd.to_numeric, errors="coerce").values.astype(np.float32)

use_bare = "diabetes" in LEAKAGE_MAP
def get_leaks(d):
    return LEAKAGE_MAP.get(d if use_bare else f"label_{d}", [])

LEAK_MASK = np.zeros((len(DISEASES), len(all_features)), dtype=bool)
for i, d in enumerate(DISEASES):
    leaked = set(get_leaks(d))
    for j, f in enumerate(all_features):
        if f in leaked: LEAK_MASK[i, j] = True

idx_tr, idx_val = train_test_split(np.arange(X_raw.shape[0]), test_size=0.2, random_state=SEED, shuffle=True)
imp = SimpleImputer(strategy="median").fit(X_raw[idx_tr])
sc  = StandardScaler().fit(imp.transform(X_raw[idx_tr]))

# Model class
class OneHealthNet(nn.Module):
    def __init__(self, n_features, n_diseases, leak_mask, d_model=96, n_heads=6, n_layers=2, dropout=0.15):
        super().__init__()
        self.n_features, self.n_diseases = n_features, n_diseases
        self.register_buffer("leak_mask", torch.tensor(leak_mask, dtype=torch.bool))
        self.tok_weight = nn.Parameter(torch.randn(n_features, d_model) * 0.02)
        self.tok_bias   = nn.Parameter(torch.zeros(n_features, d_model))
        self.mask_embed = nn.Parameter(torch.randn(d_model) * 0.02)
        el = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=d_model*4,
                                        dropout=dropout, batch_first=True, activation="gelu", norm_first=True)
        self.encoder = nn.TransformerEncoder(el, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.disease_queries = nn.Parameter(torch.randn(n_diseases, d_model) * 0.02)
        self.pool_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.heads = nn.ModuleList([nn.Linear(d_model, 1) for _ in range(n_diseases)])
        self.aux_head = nn.Linear(d_model, 1)
    def forward(self, x):
        B, F = x.shape; D = self.n_diseases
        base = x.unsqueeze(-1) * self.tok_weight.unsqueeze(0) + self.tok_bias.unsqueeze(0)
        tokens = base.unsqueeze(1).expand(B, D, F, -1).clone()
        m = self.leak_mask.unsqueeze(0).expand(B, D, F)
        tokens[m] = self.mask_embed
        tokens = tokens.reshape(B*D, F, -1)
        h = self.norm(self.encoder(tokens))
        q = self.disease_queries.unsqueeze(0).expand(B, D, -1).reshape(B*D, 1, -1)
        pooled, _ = self.pool_attn(q, h, h)
        pooled = pooled.reshape(B, D, -1)
        logits = torch.stack([self.heads[i](pooled[:, i]).squeeze(-1) for i in range(D)], dim=1)
        return logits, self.aux_head(pooled.mean(dim=1))

model = OneHealthNet(len(all_features), len(DISEASES), LEAK_MASK).to(DEVICE)
model.load_state_dict(torch.load("/kaggle/working/onehealth_net_v3_leakfix.pt", map_location=DEVICE))
model.eval()
print("✅ Loaded corrected model")

ext = pd.read_csv(f"{PROCESSED}/external_full_curated.csv")
X_ext = np.full((ext.shape[0], len(all_features)), np.nan, dtype=np.float32)
for j, f in enumerate(all_features):
    if f in ext.columns:
        X_ext[:, j] = pd.to_numeric(ext[f], errors="coerce").values
X_ext = sc.transform(imp.transform(X_ext)).astype(np.float32)
Y_ext = ext[label_cols].values.astype(np.float32)
Y_ext_present = (~np.isnan(Y_ext)).astype(np.float32)
Y_ext = np.nan_to_num(Y_ext, nan=0.0)

with torch.no_grad():
    logits, _ = model(torch.tensor(X_ext, device=DEVICE))
    prob_all = torch.sigmoid(logits).cpu().numpy()

# External AUROC for all 7 diseases
print("\n=== Corrected external validation (2021-2023) ===")
corrected_results = {}
for i, d in enumerate(DISEASES):
    lab = Y_ext_present[:, i] == 1
    a = roc_auc_score(Y_ext[lab, i], prob_all[lab, i])
    p = average_precision_score(Y_ext[lab, i], prob_all[lab, i])
    corrected_results[d] = {"auroc": float(a), "auprc": float(p)}
    print(f"  {d:22s} AUROC={a:.4f}  AUPRC={p:.4f}")

# Save
import pickle
with open("/kaggle/working/onehealth_external_v3_leakfix.pkl", "wb") as f:
    pickle.dump(corrected_results, f)
print("\n✅ Saved corrected external results")

# Permutation re-check on diabetes + metabolic_syndrome
print("\n=== Permutation re-check (leak should be gone) ===")
N_PERM = 5
for d in ["diabetes", "metabolic_syndrome"]:
    i = DISEASES.index(d)
    lab = Y_ext_present[:, i] == 1
    X_use = X_ext[lab]; y_use = Y_ext[lab, i]; base_prob = prob_all[lab, i]
    base_auroc = roc_auc_score(y_use, base_prob)
    rng = np.random.default_rng(SEED + i)
    imp_d = {}
    for j in range(X_use.shape[1]):
        drops = []
        for _ in range(N_PERM):
            Xp = X_use.copy()
            Xp[:, j] = rng.permutation(Xp[:, j])
            with torch.no_grad():
                lo, _ = model(torch.tensor(Xp, device=DEVICE))
                pp = torch.sigmoid(lo).cpu().numpy()[:, i]
            drops.append(base_auroc - roc_auc_score(y_use, pp))
        imp_d[all_features[j]] = float(np.mean(drops))
    top5 = sorted(imp_d.items(), key=lambda x: -x[1])[:5]
    print(f"\n{d} (baseline AUROC={base_auroc:.4f}) — top 5:")
    for fname, drop in top5:
        print(f"  {fname:25s} {drop:+.4f}")

/tmp/ipykernel_58/908312879.py:52: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(el, num_layers=n_layers)


✅ Loaded corrected model

=== Corrected external validation (2021-2023) ===
  diabetes               AUROC=0.8683  AUPRC=0.6332
  hypertension           AUROC=0.7996  AUPRC=0.8207
  ckd                    AUROC=0.8824  AUPRC=0.5727
  dyslipidemia           AUROC=0.7110  AUPRC=0.7688
  cvd                    AUROC=0.8461  AUPRC=0.4440
  metabolic_syndrome     AUROC=0.8497  AUPRC=0.6987
  liver_disease          AUROC=0.8144  AUPRC=0.4294

✅ Saved corrected external results

=== Permutation re-check (leak should be gone) ===

diabetes (baseline AUROC=0.8683) — top 5:
  RIDAGEYR                  +0.0711
  LBXTC                     +0.0203
  LBXIN                     +0.0152
  URDACT                    +0.0125
  LBXSCLSI                  +0.0086

metabolic_syndrome (baseline AUROC=0.8497) — top 5:
  LBXIN                     +0.0416
  BMXBMI                    +0.0270
  LBXGH                     +0.0250
  RIDAGEYR                  +0.0208
  RIAGENDR                  +0.0110


In [8]:
print("=" * 70)
print("LEAK FIX COMPLETE — Summary")
print("=" * 70)
print("\nOld (leaky) mean AUROC: 0.8342 external")
print(f"New (corrected) mean AUROC: {np.mean([m['auroc'] for m in corrected_results.values()]):.4f} external")
print("\nPer-disease comparison:")
print(f"{'Disease':22s}  {'Old':>8s}  {'New':>8s}  {'Δ':>8s}")
print("-" * 55)

OLD = {
    "diabetes": 0.9335, "hypertension": 0.7971, "ckd": 0.8759,
    "dyslipidemia": 0.7132, "cvd": 0.8415, "metabolic_syndrome": 0.8542,
    "liver_disease": 0.8239,
}
for d in DISEASES:
    old = OLD[d]
    new = corrected_results[d]["auroc"]
    print(f"{d:22s}  {old:8.4f}  {new:8.4f}  {new-old:+8.4f}")

LEAK FIX COMPLETE — Summary

Old (leaky) mean AUROC: 0.8342 external
New (corrected) mean AUROC: 0.8245 external

Per-disease comparison:
Disease                      Old       New         Δ
-------------------------------------------------------
diabetes                  0.9335    0.8683   -0.0652
hypertension              0.7971    0.7996   +0.0025
ckd                       0.8759    0.8824   +0.0065
dyslipidemia              0.7132    0.7110   -0.0022
cvd                       0.8415    0.8461   +0.0046
metabolic_syndrome        0.8542    0.8497   -0.0045
liver_disease             0.8239    0.8144   -0.0095
